# CatBoost severe-wasting model with Adaptive Greedy Search (50 evaluations)

This notebook models `severe_wasting` using the requested predictors and a complete-case dataset. It uses **Adaptive Greedy Search (AGS)** for hyperparameter tuning with a maximum budget of 50 evaluated configurations, then evaluates the selected model on a held-out test set and refits it on all eligible complete cases.

**Outcome:** `severe_wasting`. **Predictors:** 23 variables listed in Section 4.

**Important:** full-data predictions from the final refitted model are in-sample. Use the held-out test metrics for model-performance reporting; external validation is still needed before deployment.

## 1. Environment setup

Activity log: define the required packages and install any that are missing from the active Jupyter environment.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "catboost": "catboost",
    "ags": "adaptive-greedy-search==3.7.1",
    "matplotlib": "matplotlib",
}

missing_packages = [
    pip_name for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", *missing_packages
    ])
else:
    print("All required packages are already installed.")

## 2. Imports and reproducibility

Activity log: import modelling, evaluation and plotting tools, then set a fixed random seed for repeatable splitting and tuning.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ags import AdaptiveGreedySearch
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Adaptive Greedy Search:", __import__("ags").__version__)

## 3. Load the MICS child dataset

Activity log: look for the expected Stata file in the repository root. If your local filename or location differs, update `DATA_PATH` below.

In [ ]:
DATA_PATH = Path("MICS-child-ISRC.dta")
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.dta"))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
        print(f"Using discovered dataset: {DATA_PATH}")
    elif len(candidates) == 0:
        raise FileNotFoundError("No .dta file found in the current directory. Set DATA_PATH to your MICS dataset.")
    else:
        raise FileNotFoundError(f"Multiple .dta files found: {candidates}. Set DATA_PATH explicitly.")

df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)


## 4. Select the requested variables and define the outcome

Activity log: retain only the exact requested columns. `severe_wasting` is the outcome; the remaining 23 columns are predictors. Stop with a clear error if the source dataset does not contain a requested variable.

In [ ]:
TARGET = "severe_wasting"
PREDICTORS = [
    "age_m", "female", "division", "urban", "wb4", "parity_cat",
    "diarrhoea_2w", "cough_2w", "breathing_2w", "fuel_cat", "water_cat",
    "toilet_cat", "edu_level", "electricity", "wealth", "birth_order",
    "hh_size", "water_treated", "water_boil_filter", "handwash_place",
    "handwash_water", "floor_quality", "ari_chest"
]
REQUESTED_COLUMNS = [TARGET] + PREDICTORS
missing_columns = [col for col in REQUESTED_COLUMNS if col not in df.columns]
if missing_columns:
    raise KeyError(f"These requested variables are missing from the dataset: {missing_columns}")

model_df = df[REQUESTED_COLUMNS].copy()
print("Selected data shape:", model_df.shape)
print("Number of predictors:", len(PREDICTORS))
print("Outcome values before cleaning:", model_df[TARGET].value_counts(dropna=False).to_dict())

## 5. Complete-case cleaning and data checks

Activity log: drop every row with a missing value in the outcome or any requested predictor. Validate that the outcome is binary and both classes remain. No imputation is performed.

In [ ]:
rows_before = len(model_df)
missing_summary = model_df.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].to_frame("missing_rows"))
model_df = model_df.dropna(subset=REQUESTED_COLUMNS).copy()
rows_after = len(model_df)
print(f"Rows before complete-case filtering: {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Complete cases retained: {rows_after:,} ({rows_after / rows_before:.1%})")

model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="raise").astype(int)
observed_classes = sorted(model_df[TARGET].unique().tolist())
if observed_classes != [0, 1]:
    raise ValueError(f"Expected binary outcome coded 0 and 1; found {observed_classes}. Check severe_wasting coding.")

print("Complete-case outcome counts:", model_df[TARGET].value_counts().sort_index().to_dict())
print("Complete-case outcome prevalence:", model_df[TARGET].mean())


## 6. Identify categorical predictors and prepare CatBoost inputs

Activity log: declare survey-coded categories as categorical features so CatBoost does not treat their integer codes as continuous measurements. Convert categorical values to strings; keep age and household size as numeric. The list is explicit and can be adjusted if the source codebook indicates different measurement levels.

In [ ]:
CATEGORICAL_COLUMNS = [
    "female", "division", "urban", "parity_cat", "diarrhoea_2w", "cough_2w",
    "breathing_2w", "fuel_cat", "water_cat", "toilet_cat", "edu_level",
    "electricity", "wealth", "birth_order", "water_treated", "water_boil_filter",
    "handwash_place", "handwash_water", "floor_quality", "ari_chest"
]
NUMERIC_COLUMNS = [col for col in PREDICTORS if col not in CATEGORICAL_COLUMNS]

def prepare_catboost_frame(frame):
    """Return a copy with categorical columns represented consistently as strings."""
    out = frame.copy()
    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].astype(str)
    for col in NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="raise")
    return out

X = prepare_catboost_frame(model_df[PREDICTORS])
y = model_df[TARGET].copy()
CAT_FEATURE_INDICES = [X.columns.get_loc(col) for col in CATEGORICAL_COLUMNS]
print("Categorical predictors:", CATEGORICAL_COLUMNS)
print("Numeric predictors:", NUMERIC_COLUMNS)
print("Categorical feature indices:", CAT_FEATURE_INDICES)


## 7. Stratified train/test split

Activity log: reserve 20% of complete cases as a held-out test set and preserve the severe-wasting proportion in both partitions. Optuna will only see the training partition.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Training rows:", len(X_train), "| severe-wasting cases:", int(y_train.sum()))
print("Test rows:", len(X_test), "| severe-wasting cases:", int(y_test.sum()))


## 8. Define the CatBoost pipeline and AGS search space

Activity log: build a scikit-learn pipeline with CatBoost and define a finite, discrete hyperparameter grid. AGS evaluates configurations using stratified five-fold cross-validation and average precision (PR-AUC), which is appropriate for an uncommon outcome. CatBoost uses one CPU thread per fit to avoid oversubscription during parallel AGS evaluations.

In [ ]:
def make_pipeline(iterations=2500):
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        iterations=iterations,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
        cat_features=CAT_FEATURE_INDICES,
    )
    return Pipeline([
        ("prepare", FunctionTransformer(prepare_catboost_frame, validate=False)),
        ("catboost", model),
    ])

# AGS expects a finite grid. The grid is intentionally broad and uses
# denser choices for the most influential CatBoost hyperparameters.
AGS_PARAM_GRID = {
    "catboost__depth": [4, 5, 6, 7, 8, 9, 10],
    "catboost__learning_rate": [0.01, 0.02, 0.04, 0.06, 0.08, 0.12, 0.16, 0.20],
    "catboost__l2_leaf_reg": [0.1, 0.3, 1.0, 3.0, 10.0, 20.0, 30.0],
    "catboost__random_strength": [0.001, 0.01, 0.1, 1.0, 5.0, 10.0],
    "catboost__bagging_temperature": [0.0, 0.5, 1.0, 2.0, 3.0, 5.0],
    "catboost__border_count": [64, 96, 128, 192, 254],
    "catboost__auto_class_weights": [None, "Balanced", "SqrtBalanced"],
}

print("Search-space upper bound:",
      int(np.prod([len(v) for v in AGS_PARAM_GRID.values()])))
print("Maximum AGS evaluation budget: 50")

## 9. Run Adaptive Greedy Search (maximum 50 evaluations)

Activity log: tune CatBoost with AGS using a maximum of 50 unique hyperparameter configurations. Each completed configuration is evaluated by stratified five-fold cross-validation. Confidence-based racing can stop clearly weaker configurations before all folds are run. The optimization objective is average precision (PR-AUC).

In [ ]:
base_pipeline = make_pipeline()

ags_search = AdaptiveGreedySearch(
    estimator=base_pipeline,
    param_grid=AGS_PARAM_GRID,
    cv=5,
    scoring="average_precision",
    max_evaluations=50,
    n_climbers=4,
    n_jobs=-1,
    random_state=SEED,
    surrogate_type="rf",
    acquisition="ucb",
    pruning_strategy="racing",
    racing_confidence=0.95,
    candidate_pool_size=256,
    profiling=False,
)

ags_search.fit(X_train, y_train)

print("AGS evaluations:", ags_search.n_evaluations)
print("Completed full-CV evaluations:", ags_search.n_completed_evaluations)
print("Partially evaluated/pruned configurations:", ags_search.n_partial_evaluations)
print("Folds saved by racing:", ags_search.folds_saved)
print("Best 5-fold average precision:", ags_search.best_score)
print("Best CatBoost hyperparameters:")
print(json.dumps(ags_search.best_params, indent=2))

best_pipeline = make_pipeline()
best_pipeline.set_params(**ags_search.best_params)
best_pipeline.fit(X_train, y_train)

## 10. Review AGS search history

Activity log: inspect the configurations evaluated by AGS, compare completed cross-validation scores, and plot the best-so-far average-precision trajectory. Pruned configurations are retained in the history with their partial-fold information.

In [ ]:
ags_history_df = pd.DataFrame(ags_search.history)
display(
    ags_history_df.sort_values(
        "score", ascending=False, na_position="last"
    ).head(10)
)

completed_history = ags_history_df[ags_history_df["completed"]].copy()
completed_history = completed_history.reset_index(drop=True)
completed_history["evaluation"] = np.arange(1, len(completed_history) + 1)
completed_history["best_so_far"] = completed_history["score"].cummax()

plt.figure(figsize=(9, 5))
plt.plot(
    completed_history["evaluation"],
    completed_history["score"],
    marker="o",
    alpha=0.7,
    label="Completed CV score"
)
plt.plot(
    completed_history["evaluation"],
    completed_history["best_so_far"],
    linewidth=2,
    label="Best-so-far PR-AUC"
)
plt.xlabel("Completed AGS evaluation")
plt.ylabel("Mean 5-fold average precision")
plt.title("Adaptive Greedy Search history — CatBoost")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Evaluate the best AGS-tuned pipeline on the held-out test set

Activity log: fit the best AGS configuration on the training partition and calculate discrimination and classification metrics on the untouched test partition. The 0.50 threshold is reported for transparency; it is not necessarily the optimal threshold for a rare outcome.

In [ ]:
best_pipeline = make_pipeline()
best_pipeline.set_params(**ags_search.best_params)
best_pipeline.fit(X_train, y_train)

test_probability = best_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

test_metrics = {
    "n_test": int(len(y_test)),
    "severe_wasting_cases": int(y_test.sum()),
    "roc_auc": float(roc_auc_score(y_test, test_probability)),
    "pr_auc_average_precision": float(average_precision_score(y_test, test_probability)),
    "accuracy": float(accuracy_score(y_test, test_prediction)),
    "precision": float(precision_score(y_test, test_prediction, zero_division=0)),
    "recall": float(recall_score(y_test, test_prediction, zero_division=0)),
    "f1": float(f1_score(y_test, test_prediction, zero_division=0)),
}
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}))
print(classification_report(y_test, test_prediction, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, test_prediction))

RocCurveDisplay.from_predictions(y_test, test_probability)
plt.title("CatBoost ROC curve — held-out test set")
plt.tight_layout()
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, test_probability)
plt.title("CatBoost precision-recall curve — held-out test set")
plt.tight_layout()
plt.show()

## 12. Refit the best configuration on the overall complete-case dataset

Activity log: after recording the held-out evaluation, refit the best AGS configuration using every eligible complete-case observation. This produces the final model requested for the overall analysis. Predictions from this refitted model are in-sample and must not be presented as independent validation results.

In [ ]:
final_pipeline = make_pipeline()
final_pipeline.set_params(**ags_search.best_params)
final_pipeline.fit(X, y)

model_df["predicted_probability_severe_wasting"] = final_pipeline.predict_proba(X)[:, 1]
model_df["predicted_class_severe_wasting"] = (
    model_df["predicted_probability_severe_wasting"] >= 0.5
).astype(int)

print("Final model refitted on all complete cases:", len(model_df))
print("Observed severe-wasting cases:", int(y.sum()))
print("Mean fitted probability:",
      model_df["predicted_probability_severe_wasting"].mean())
display(model_df[
    [TARGET, "predicted_probability_severe_wasting",
     "predicted_class_severe_wasting"]
].head())

## 13. Save model outputs and reproducibility information

Activity log: save the AGS evaluation history, held-out test metrics, full-data predictions, best hyperparameters and run manifest under `outputs/catboost_ags/`. The notebook itself does not commit generated outputs; run it locally to create them.

In [ ]:
OUTPUT_DIR = Path("outputs") / "catboost_ags"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ags_history_df.to_csv(OUTPUT_DIR / "ags_search_history.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(
    OUTPUT_DIR / "heldout_test_metrics.csv", index=False
)
model_df.to_csv(OUTPUT_DIR / "complete_case_predictions.csv", index=False)

with open(OUTPUT_DIR / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(ags_search.best_params, f, indent=2)

manifest = {
    "data_path": str(DATA_PATH),
    "seed": SEED,
    "target": TARGET,
    "predictors": PREDICTORS,
    "categorical_predictors": CATEGORICAL_COLUMNS,
    "rows_before_complete_case_filter": int(rows_before),
    "complete_case_rows": int(len(model_df)),
    "complete_case_events": int(y.sum()),
    "optimizer": "AdaptiveGreedySearch",
    "ags_max_evaluations": 50,
    "ags_evaluations": int(ags_search.n_evaluations),
    "ags_completed_evaluations": int(ags_search.n_completed_evaluations),
    "ags_partial_evaluations": int(ags_search.n_partial_evaluations),
    "ags_folds_saved": int(ags_search.folds_saved),
    "optimization_metric": "average_precision (5-fold CV PR-AUC)",
    "best_cv_score": float(ags_search.best_score),
    "best_hyperparameters": ags_search.best_params,
    "heldout_test_metrics": test_metrics,
    "note": "Full-data predictions are in-sample after refitting on all complete cases; use held-out test metrics for evaluation."
}
with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())

## 14. Interpretation and cautions

- AGS uses a finite, discrete hyperparameter grid and a maximum budget of 50 evaluated configurations. It is a heuristic search method and does not guarantee the global optimum.
- The tuning objective is mean five-fold average precision (PR-AUC), selected because severe wasting is an uncommon outcome.
- ROC-AUC measures ranking discrimination, while precision and recall at the 0.50 threshold may be unhelpful for a rare outcome. Threshold selection should be performed using training/validation data if an operational threshold is needed.
- Complete-case deletion can change the analytic population and introduce selection bias. Report the number and percentage of observations removed and compare included vs excluded records where possible.
- CatBoost feature contributions and predictions are associational, not causal.
- The full-data fitted probabilities are not out-of-sample predictions. Use a separate external dataset or appropriately designed cross-validation for generalization claims.

## 15. Final evaluation summary and SHAP interpretationActivity log: calculate precision, recall, F1-score, ROC-AUC and PR-AUC from the held-out test predictions, plot the ROC curve, and calculate CatBoost-native SHAP contributions for the held-out test observations. The SHAP bar chart ranks predictors by their mean absolute contribution; it does not indicate causality or direction of effect.

In [ ]:
# ============================================================# FINAL CELL: HELD-OUT METRICS, ROC CURVE AND SHAP ANALYSIS# ============================================================# 1. Held-out test predictions (best AGS-tuned model)final_test_probability = best_pipeline.predict_proba(X_test)[:, 1]THRESHOLD = 0.50final_test_prediction = (final_test_probability >= THRESHOLD).astype(int)# 2. Classification metricsfinal_metrics = {    "Precision": precision_score(y_test, final_test_prediction, zero_division=0),    "Recall": recall_score(y_test, final_test_prediction, zero_division=0),    "F1 Score": f1_score(y_test, final_test_prediction, zero_division=0),    "ROC-AUC": roc_auc_score(y_test, final_test_probability),    "PR-AUC": average_precision_score(y_test, final_test_probability),    "Accuracy": accuracy_score(y_test, final_test_prediction),}final_metrics_df = pd.DataFrame(    final_metrics.items(), columns=["Metric", "Score"])print("HELD-OUT TEST PERFORMANCE (threshold = 0.50)")display(final_metrics_df.style.format({"Score": "{:.4f}"}))print("\nClassification report:")print(classification_report(    y_test, final_test_prediction,    target_names=["Not Severe Wasting", "Severe Wasting"],    zero_division=0))print("Confusion matrix:")print(confusion_matrix(y_test, final_test_prediction))# 3. ROC curvefpr, tpr, roc_thresholds = roc_curve(y_test, final_test_probability)roc_auc_value = roc_auc_score(y_test, final_test_probability)plt.figure(figsize=(8, 6))plt.plot(fpr, tpr, linewidth=2,         label=f"CatBoost + AGS (AUC = {roc_auc_value:.4f})")plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")plt.xlabel("False Positive Rate")plt.ylabel("True Positive Rate (Recall)")plt.title("ROC Curve — Severe Wasting (Held-out Test Set)")plt.legend(loc="lower right")plt.grid(alpha=0.25)plt.tight_layout()plt.show()# 4. CatBoost-native SHAP values (no separate shap package required)catboost_model = best_pipeline.named_steps["catboost"]X_test_shap = prepare_catboost_frame(X_test)shap_pool = Pool(    data=X_test_shap,    label=y_test,    cat_features=CATEGORICAL_COLUMNS)shap_values_all = catboost_model.get_feature_importance(    data=shap_pool,    type="ShapValues")# Last column contains the expected/base value; exclude it from feature effects.shap_contributions = shap_values_all[:, :-1]# 5. Global SHAP importanceshap_importance_df = pd.DataFrame({    "Variable": X_test_shap.columns,    "Mean Absolute SHAP": np.abs(shap_contributions).mean(axis=0)}).sort_values(    "Mean Absolute SHAP", ascending=False).reset_index(drop=True)print("\nGLOBAL SHAP FEATURE IMPORTANCE")display(shap_importance_df.style.format({    "Mean Absolute SHAP": "{:.6f}"}))# 6. SHAP importance bar graphtop_n = min(20, len(shap_importance_df))bar_data = shap_importance_df.head(top_n).sort_values(    "Mean Absolute SHAP", ascending=True)plt.figure(figsize=(10, 8))plt.barh(bar_data["Variable"], bar_data["Mean Absolute SHAP"])plt.xlabel("Mean Absolute SHAP Value")plt.ylabel("Predictor")plt.title(f"CatBoost SHAP Importance — Top {top_n} Predictors")plt.grid(axis="x", alpha=0.25)plt.tight_layout()plt.show()# 7. SHAP beeswarm-style distribution plot for the most important predictors.# Point colour represents the within-variable feature-value rank.top_features = shap_importance_df.head(min(15, len(shap_importance_df)))["Variable"].tolist()feature_to_index = {name: i for i, name in enumerate(top_features)}rng = np.random.default_rng(SEED)plt.figure(figsize=(11, 8))for row, feature in enumerate(top_features):    col_idx = X_test_shap.columns.get_loc(feature)    shap_x = shap_contributions[:, col_idx]    raw_values = X_test_shap[feature]    numeric_values = pd.to_numeric(raw_values, errors="coerce")    if numeric_values.notna().sum() == 0:        encoded_values = pd.Series(            pd.factorize(raw_values.astype(str), sort=True)[0],            index=raw_values.index,            dtype=float        )    else:        encoded_values = numeric_values.astype(float)        if encoded_values.isna().any():            encoded_values = encoded_values.fillna(encoded_values.median())    value_min = encoded_values.min()    value_max = encoded_values.max()    if pd.isna(value_min) or pd.isna(value_max) or value_min == value_max:        colour_values = np.full(len(encoded_values), 0.5)    else:        colour_values = (            (encoded_values.to_numpy() - value_min) / (value_max - value_min)        )    jitter = rng.uniform(-0.28, 0.28, size=len(shap_x))    plt.scatter(        shap_x,        row + jitter,        c=colour_values,        cmap="coolwarm",        vmin=0,        vmax=1,        s=18,        alpha=0.65,        edgecolors="none"    )plt.axvline(0, color="black", linewidth=1, linestyle="--")plt.yticks(range(len(top_features)), top_features)plt.gca().invert_yaxis()plt.xlabel("SHAP value (impact on model output)")plt.ylabel("Predictor")plt.title("SHAP Summary — Beeswarm-style Distribution (Top Predictors)")colourbar = plt.colorbar()colourbar.set_label("Relative feature value (low to high)")plt.tight_layout()plt.show()# 8. Save final analysis outputsOUTPUT_DIR.mkdir(parents=True, exist_ok=True)final_metrics_df.to_csv(    OUTPUT_DIR / "final_evaluation_metrics.csv", index=False)shap_importance_df.to_csv(    OUTPUT_DIR / "shap_feature_importance.csv", index=False)pd.DataFrame(    shap_contributions, columns=X_test_shap.columns).to_csv(    OUTPUT_DIR / "test_set_shap_values.csv", index=False)print("\nFinal evaluation and SHAP analysis completed.")print("Files saved in:", OUTPUT_DIR.resolve())